In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum, avg, min, max, mean,
    when, trim, to_timestamp
)
from pyspark.sql.types import TimestampType

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum, avg, min, max, mean
from pyspark.sql.types import TimestampType

In [3]:
spark = SparkSession.builder \
    .appName("Week5_PySpark_Assignment") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session Created Successfully")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/19 21:53:00 WARN Utils: Your hostname, Tanvis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/07/19 21:53:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/19 21:53:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully


In [4]:
df = spark.read.csv(
    "data/data.csv",
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully")

Dataset loaded successfully


In [5]:
df.show(5)

+-------------------+----------+--------------------+--------------------+---------+-----+--------------+---------------+--------------+--------------------+----------------+-----------------+-----------+----------+----------+------------+-------------+--------------+---------+----------+----------+-----------+-------------+---------+
|           Order No|Order Date|       Customer Name|             Address|     City|State| Customer Type|Account Manager|Order Priority|        Product Name|Product Category|Product Container|  Ship Mode| Ship Date|Cost Price|Retail Price|Profit Margin|Order Quantity|Sub Total|Discount %|Discount $|Order Total|Shipping Cost|    Total|
+-------------------+----------+--------------------+--------------------+---------+-----+--------------+---------------+--------------+--------------------+----------------+-----------------+-----------+----------+----------+------------+-------------+--------------+---------+----------+----------+-----------+-------------+

In [6]:
df.printSchema()
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

root
 |-- Order No: timestamp (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Customer Type: string (nullable = true)
 |-- Account Manager: string (nullable = true)
 |-- Order Priority: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Product Category: string (nullable = true)
 |-- Product Container: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Cost Price: string (nullable = true)
 |-- Retail Price: string (nullable = true)
 |-- Profit Margin: string (nullable = true)
 |-- Order Quantity: integer (nullable = true)
 |-- Sub Total: string (nullable = true)
 |-- Discount %: string (nullable = true)
 |-- Discount $: string (nullable = true)
 |-- Order Total: string (nullable = true)
 |-- Shipping Cost: string (nullable = true)
 

### Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Traditional MapReduce has several limitations:

- It writes intermediate results to disk after every processing stage, which increases disk I/O and slows down execution.
- It is inefficient for iterative algorithms because the same data must be repeatedly read from disk.
- It is less suitable for interactive analytics and real-time processing.
- Writing complex workflows often requires multiple MapReduce jobs, making development more difficult.

Spark is preferred because it supports in-memory processing, DAG-based execution, faster iterative computation, and high-level APIs such as DataFrames and Spark SQL.

### Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark can keep frequently reused datasets in memory using caching or persistence. In iterative machine learning algorithms, the same dataset is processed multiple times. Instead of reading and writing the data to disk during every iteration, Spark reuses the data directly from memory.

This reduces disk I/O and significantly improves processing speed compared to traditional disk-based systems such as MapReduce.

### Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.
This removes duplicate records that have the same `user_id` and `transaction_date`, while keeping one occurrence of each unique combination.

In [7]:
df_clean = df.dropDuplicates(["Order No", "Order Date"])

print("Rows before removing duplicates:", df.count())
print("Rows after removing duplicates:", df_clean.count())

df_clean.show(5)

Rows before removing duplicates: 5000
Rows after removing duplicates: 4355


26/07/19 21:54:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------+----------+--------------------+--------------------+---------+-----+--------------+---------------+--------------+--------------------+----------------+-----------------+-----------+----------+----------+------------+-------------+--------------+---------+----------+----------+-----------+-------------+---------+
|           Order No|Order Date|       Customer Name|             Address|     City|State| Customer Type|Account Manager|Order Priority|        Product Name|Product Category|Product Container|  Ship Mode| Ship Date|Cost Price|Retail Price|Profit Margin|Order Quantity|Sub Total|Discount %|Discount $|Order Total|Shipping Cost|    Total|
+-------------------+----------+--------------------+--------------------+---------+-----+--------------+---------------+--------------+--------------------+----------------+-----------------+-----------+----------+----------+------------+-------------+--------------+---------+----------+----------+-----------+-------------+

### Q4 Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.
This filters records for Western Australia (`WA`), groups them by product category, and calculates the average retail price for each category.

In [8]:

from pyspark.sql.functions import avg, regexp_replace, col

df_q4 = df.withColumn(
    "Retail Price Numeric",
    regexp_replace(col("Retail Price"), r"[$,]", "").cast("double")
)

result_q4_actual = (
    df_q4
    .filter(col("State") == "WA")
    .groupBy("Product Category")
    .agg(avg("Retail Price Numeric").alias("Average Retail Price"))
)

result_q4_actual.show()

+----------------+--------------------+
|Product Category|Average Retail Price|
+----------------+--------------------+
+----------------+--------------------+



### Q5. Difference between `.na.drop()` and `.na.fill()`

`.na.drop()` removes rows containing null values, while `.na.fill()` replaces null values with a specified value.

Using `.na.fill()` is useful when we want to preserve the row instead of deleting it.

In [ ]:
df_filled = df.na.fill({"status": "Unknown"})

### Q6: Count of records for each city where count is greater than 100

In [9]:
result_q6 = df.groupBy("City").count().filter(col("count") > 100)

result_q6.show()

+---------+-----+
|     City|count|
+---------+-----+
|   Sydney| 3584|
|Melbourne| 1416|
+---------+-----+



### Q7: Immutability of Spark DataFrames

Spark DataFrames are immutable, which means the original DataFrame cannot be modified directly. Operations such as dropping a column or renaming a column create a new DataFrame instead of changing the existing one.

Therefore, during data cleaning, the transformed DataFrame must be assigned to a new variable or reassigned to the existing variable.

### Q8: Filter rows based on age and subscription

In [ ]:
filtered_df = df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

### Q9: Handling null values before mathematical aggregations

It is better to handle null values before performing mathematical aggregations because missing values can result in incomplete or misleading analysis. Cleaning null values first ensures that operations such as sum() and avg() are performed on consistent and valid data.

Depending on the meaning of the data, null values can be removed or replaced with an appropriate value before aggregation.

### Q10: Cast and rename a timestamp column

In [ ]:
from pyspark.sql.types import TimestampType

df_updated = df.withColumn(
    "raw_timestamp",
    col("raw_timestamp").cast(TimestampType())
).withColumnRenamed(
    "raw_timestamp",
    "event_time"
)

### Q11: Shuffle process during grouping

During a grouping operation such as `groupBy()`, Spark redistributes data across partitions so that rows with the same key are brought together.

This movement of data between partitions is called a shuffle. It is considered a wide transformation because one input partition can contribute data to multiple output partitions.

### Q12: Remove rows with null email or empty username

In [ ]:
clean_df = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

### Q13: Calculate min, max, and mean using `.agg()`

In [ ]:
result_q13 = df.agg(
    min("price").alias("minimum_price"),
    max("price").alias("maximum_price"),
    mean("price").alias("average_price")
)

result_q13.show()

### Q14: Risk of using `inferSchema=true` with inconsistent date formats

When a dataset contains inconsistent or messy date formats, `inferSchema=true` may incorrectly infer the column type or treat the date column as a string.

This can lead to failed conversions, incorrect comparisons, and inaccurate analysis. Defining the schema explicitly is safer when the source data is inconsistent.

### Q15: Final data processing pipeline

In [ ]:
final_df = (
    df
    .dropDuplicates()
    .na.fill({"price": 0})
    .groupBy("store_id")
    .agg(sum("price").alias("total_revenue"))
)

final_df.show()